# Comunicación y Visualizaciones
Python para Ciencia de Datos | UADE
Dataset: Airbnb Buenos Aires (alojamientos + hosts)

Para esta clase vamos a trabajar con los mismos datasets de la **actividad integradora de la U5**: `airbnb_jr` (alojamientos) y `airbnb_hosts` (anfitriones).

Vamos a ver tres tipos de gráficos de barras:

| Tipo | ¿Para qué sirve? | Librería |
|---|---|---|
| **Simple** | comparar una métrica entre categorías | seaborn |
| **Agrupado** | comparar dos o más métricas lado a lado | seaborn (`hue`) |
| **Apilado** | ver cómo se reparte un total entre partes | matplotlib (`bottom`) |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Cargar datasets

In [ ]:
# Dataset de alojamientos
alojamientos = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_jr.csv")
print(f"Dataset airbnb_jr loaded {alojamientos.shape}")

# Dataset de hosts
hosts = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_hosts.csv")
print(f"Dataset airbnb_hosts loaded {hosts.shape}")

## Limpieza y merge
Repetimos la misma limpieza que en la actividad integradora de la U5.

In [ ]:
# Normalizar nombres de columnas
alojamientos.columns = alojamientos.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

# Eliminar duplicados y nulos, y renombrar neighbourhood -> barrio
alojamientos.drop_duplicates(inplace=True)
alojamientos.dropna(inplace=True)
alojamientos.rename(columns={"neighbourhood": "barrio"}, inplace=True)

# Convertimos price de texto ("$86,113.00") a número, y lo pasamos a dólares
alojamientos["price"] = alojamientos["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
alojamientos["price"] = pd.to_numeric(alojamientos["price"], errors="coerce")


# Convertimos last_review a datetime
alojamientos["last_review"] = pd.to_datetime(alojamientos["last_review"], format="%Y-%m-%d", errors="coerce")

# Hosts: eliminar duplicados
hosts.drop_duplicates(inplace=True)

In [ ]:
# Left join: conservamos todos los alojamientos y les sumamos los datos de su host
df = pd.merge(alojamientos, hosts, on="host_id", how="left", validate="m:1")
# df.head(2)

## Feature Engineering

In [ ]:
def clasificar_location(location):
    if pd.isna(location):
        return "sin dato"
    elif "Buenos Aires" in location:
        return "local"
    else:
        return "extranjero / otra ciudad"

In [ ]:
# Crear price_usd a partir de price
df["price_usd"] = round(df["price"] / 1500, 2)

# Extraer year y month de last_review
df["last_review_year"] = df["last_review"].dt.year
df["last_review_month"] = df["last_review"].dt.month

# Crear host_origen a partir de host_location
df["host_type"] = df["host_location"].apply(clasificar_location)

# Crear host_tiene_about a partir de host_about
df["host_tiene_about"] = hosts["host_about"].notna() & (hosts["host_about"].str.strip() != "-")

# Crear host_activo a partir de last_review
fecha_ref = df["last_review"].max()
df["host_activo"] = df["last_review"] >= fecha_ref - pd.DateOffset(months=12)


In [ ]:
# Estructura y tipos de datos
df.info()

# Visualizaciones

## Tipo de alojamiento y precios
Comencemos por graficar **cuántos alojamientos hay de cada tipo de habitación** (`room_type`) y el precio promedio por cada tipo

Primero preparamos los datos: un gráfico de barras necesita **una fila por categoría**, así que agrupamos.

In [ ]:
# Agrupación/agregación:
# Queremos la cantidad de alojamientos y el precio promedio, agrupados por tipo de habitación
df_room = df.groupby("room_type").agg(
    alojamientos=("host_id", "count"),
    precio_avg=("price_usd", "mean"),
).reset_index()
df_room

In [ ]:
# Generamos un gráfico de barras simple, mostrando la cantidad de alojamientos por tipo

# Estilo y paleta
sns.set_theme(style="whitegrid", palette="Set2")
plt.figure(figsize=(12, 4))

# Subplot 1 - alojamientos
plt.subplot(1, 2, 1)
sns.barplot(data=df_room, x="room_type", y="alojamientos")
plt.title("Cantidad de alojamientos por tipo")

# Subplot 2 - precio promedio
plt.subplot(1, 2, 2)
sns.barplot(data=df_room, x="room_type", y="precio_avg")
plt.title("Precio promedio (USD) por tipo")

# Personalización del gráfico
plt.suptitle("Alojamientos y precios por tipo de habitación", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## Barrios y precios

Actividad: utilizar el código arriba para graficar cantidad de alojamientos por barrio y precio promedio por barrio

In [ ]:
# Tu código aqui:

# Agrupación/agregación:
# Queremos la cantidad de alojamientos y el precio promedio, agrupados por tipo de habitación
df_barrio = df.groupby("barrio").agg(
    alojamientos=("host_id", "count"),
    precio_avg=("price_usd", "mean"),
)
df_barrio = df_barrio.sort_values(by="alojamientos", ascending=False).head(5).reset_index()

In [ ]:
# Generamos un gráfico de barras simple, 
# mostrando la cantidad de alojamientos y precio promedio por tipo

# Estilo y paleta
sns.set_theme(style="whitegrid", palette="Set2")
plt.figure(figsize=(12, 4))

# Subplot 1 - alojamientos
plt.subplot(1, 2, 1)
sns.barplot(data=df_barrio.sort_values(by="alojamientos", ascending=False), 
            x="barrio", 
            y="alojamientos")
plt.title("Cantidad de alojamientos por barrio")

# Subplot 2 - precio promedio
plt.subplot(1, 2, 2)
sns.barplot(data=df_barrio.sort_values(by="precio_avg", ascending=False), 
            x="barrio", 
            y="precio_avg")
plt.title("Precio promedio (USD) por barrio")


# Personalización del gráfico
plt.suptitle("Alojamientos y precios por barrio", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## Alojamientos activos e inactivos en el último año
**Gráfico de barras apilado**
<br>
Queremos ver, para cada barrio, cuántos alojamientos están **activos** y cuántos **inactivos**. Activos + inactivos = total, así que tiene sentido **apilarlos**: la altura total de la barra es la cantidad de alojamientos del barrio.

**Nota:** seaborn no soporta de manera nativa el apilamiento ("stack"), por lo que hay que graficar con matplotlib.

In [ ]:
# Realizamos la agrupación y agregación
df_barrio = df.groupby("barrio").agg(
    alojamientos = ("host_id", "count"),
    activos = ("host_activo", "sum")
).sort_values(by="alojamientos", ascending=False).head(10).reset_index()

# Calculamos los inactivos como la diferencia de alojamientos y activos
df_barrio["inactivos"] = df_barrio["alojamientos"] - df_barrio["activos"]

In [ ]:
# Seaborn no soporta stack, asi que usamos matplot

sns.set_theme(style="whitegrid", palette="Set2")
plt.figure(figsize=(10,6))

# Primer nivel (abajo)
plt.bar(df_barrio["barrio"], 
        df_barrio["activos"], 
        label="Activos")   # plt.algo es matplotlib

# Segundo nivel (arriba)
# bottom indica dónde empieza la barra: arriba de los activos
plt.bar(df_barrio["barrio"], 
        df_barrio["inactivos"], bottom=df_barrio["activos"], label="Inactivos")

# Ticks (valores de eje)
plt.tick_params(axis="y", labelsize=10)
plt.tick_params(axis="x", rotation=45, labelsize=10)

# Ajustes de título y ejes
plt.title("Alojamientos activos e inactivos por barrio", fontsize=14)
plt.xlabel("Barrio")
plt.ylabel("Cantidad de alojamientos")
plt.legend()
plt.tight_layout()
plt.show()

✍️ **Para responder:** ¿Qué barrio tiene la mayor proporción de alojamientos inactivos? ¿Es fácil verlo en este gráfico, o convendría graficar porcentajes en lugar de cantidades?

# Rango de precios por barrio

**Barras agrupadas**

In [ ]:
# Realizamos la agrupación y agregación
df_barrio = df.groupby("barrio").agg(
    alojamientos = ("host_id", "count"),
    precio_min_usd = ("price_usd", "min"),
    precio_median_usd = ("price_usd", "median"),
    precio_max_usd = ("price_usd", "max"),
)
df_barrio = df_barrio.sort_values(by="precio_median_usd", ascending=True).head(5).reset_index()

# df_barrio.describe()

Para poder representar estos datos en un gráfico de barras agrupado, necesitamos ajustar el dataframe, convirtiendo el formato `wide` a `long`. Para ello usamos `melt`.

Nota: originalmente, el dataframe tenía formato long, pero cuando agregamos, el nuevo dataframe tiene formato wide.

In [ ]:
# Convertimos el DataFrame a formato largo
df_barrio_long = df_barrio.melt(
    id_vars="barrio",
    value_vars=["precio_min_usd", "precio_median_usd", "precio_max_usd"],
    var_name="metrica",
    value_name="precio_usd"
)
# df_barrio_long.sort_values("barrio").head()

In [ ]:
# Estilo y paleta
sns.set_theme(style="whitegrid", palette="Set2")
plt.figure(figsize=(10,6))

# Barras agrupadas: hue define el color (una barra por métrica)
sns.barplot(
    data=df_barrio_long,
    x="barrio",
    y="precio_usd",
    hue="metrica"
)

# Personalización
plt.title("Precio promedio vs mediano por barrio", fontsize=14, weight="bold")
plt.xlabel("Barrio", fontsize=10)
plt.ylabel("Precio (USD)", fontsize=10)
plt.tick_params(axis="x", rotation=45, labelsize=10)
plt.legend(title="Indicador")
plt.tight_layout()
plt.show()

---
# Parte 2 — Más allá de las barras
Hasta ahora graficamos relaciones del tipo **categoría → un número** (barrio → cantidad, barrio → precio mediano). Para eso el gráfico de barras es ideal.

Pero muchas preguntas involucran **variables numéricas** o **más de dos variables**. Ahí una barra (que resume todo en un solo número) esconde información, y necesitamos otros gráficos. Son los que suelen verse en herramientas como **Flourish**.

| Relación | Tipo de variables | Gráfico |
|---|---|---|
| Barrio × host local | categoría × categoría | barras apiladas al **100%** |
| Barrio × antigüedad del host | categoría × número | **boxplot** y **beeswarm** |
| Dormitorios × precio × barrio | 3 variables | barras agrupadas y **heatmap** |
| Precio × reseñas × puntaje | número × número × número | **scatterplot** con color |

In [ ]:
# Para que los gráficos sean legibles, trabajamos con los 10 barrios con más alojamientos
top_barrios = df["barrio"].value_counts().head(10).index
df_top = df[df["barrio"].isin(top_barrios)].copy()
df_top.shape

## 1. Barrio × host local
Queremos saber qué proporción de los alojamientos de cada barrio pertenece a hosts que viven en Buenos Aires.

⚠️ `host_location` tiene valores vacíos. Si usamos solo `True/False`, los vacíos quedarían como "no local" sin serlo. Por eso creamos **tres categorías**.

In [ ]:
# Creamos la variable host_origen con tres categorías
def clasificar_origen(ubicacion):
    if pd.isna(ubicacion):
        return "sin dato"
    elif "Buenos Aires" in ubicacion:
        return "local"
    else:
        return "extranjero / otra ciudad"

df_top["host_origen"] = df_top["host_location"].apply(clasificar_origen)
df_top["host_origen"].value_counts()

Con `pd.crosstab` contamos cuántos alojamientos hay en cada combinación barrio × origen. Con `normalize="index"` cada fila suma 1, es decir, obtenemos **proporciones dentro de cada barrio**.

In [ ]:
df_origen = pd.crosstab(df_top["barrio"], df_top["host_origen"], normalize="index") * 100
df_origen = df_origen.sort_values(by="local", ascending=True).round(1)
df_origen

Usamos un **apilado al 100%**. Si apiláramos cantidades, Palermo (242 alojamientos) aplastaría a los barrios chicos y no podríamos comparar proporciones.

Esta vez lo hacemos con **barras horizontales** (`plt.barh`). En lugar de `bottom`, el parámetro para apilar es `left`.

In [ ]:
sns.set_theme(style="whitegrid", palette="Set2")
plt.figure(figsize=(10,6))

# Primer nivel (izquierda)
plt.barh(df_origen.index, df_origen["local"], label="Local")

# Segundo nivel: empieza donde termina el primero
plt.barh(df_origen.index, df_origen["extranjero / otra ciudad"],
         left=df_origen["local"], label="Extranjero / otra ciudad")

# Tercer nivel: empieza donde terminan los dos anteriores
plt.barh(df_origen.index, df_origen["sin dato"],
         left=df_origen["local"] + df_origen["extranjero / otra ciudad"], label="Sin dato", color="lightgray")

plt.title("Origen del host por barrio (% de alojamientos)", fontsize=14, weight="bold")
plt.xlabel("% de alojamientos")
plt.ylabel("")
plt.xlim(0, 100)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

💡 **Atajo:** pandas puede hacer el apilado directamente con `df_origen.plot(kind="barh", stacked=True)`, sin calcular `left` a mano.

✍️ **Para responder:** ¿En qué barrio tienen más peso los hosts que no viven en Buenos Aires?

## 2. Barrio × antigüedad del host
La antigüedad (`hosts_time_as_user_years`) es **numérica**. Veamos primero qué pasaría con un barplot de la mediana:

In [ ]:
df_top.groupby("barrio")["hosts_time_as_user_years"].median().sort_values()

Las medianas son muy parecidas (entre 6 y 9 años). Un barplot mostraría barras casi iguales y concluiríamos que "no hay diferencias", pero estaríamos escondiendo **cómo se distribuyen** los valores dentro de cada barrio.

Para que se lea mejor, en esta sección usamos los **6 barrios** con más alojamientos.

In [ ]:
top6 = df["barrio"].value_counts().head(6).index
df_top6 = df[df["barrio"].isin(top6)]

# Ordenamos los barrios por mediana de antigüedad
orden = df_top6.groupby("barrio")["hosts_time_as_user_years"].median().sort_values().index
orden

### Boxplot
El boxplot muestra en un solo gráfico la mediana, los cuartiles (la caja) y los valores extremos.

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_top6, x="barrio", y="hosts_time_as_user_years", order=orden, color="steelblue")

plt.title("Antigüedad del host por barrio", fontsize=14, weight="bold")
plt.xlabel("Barrio")
plt.ylabel("Años como usuario de Airbnb")
plt.tight_layout()
plt.show()

### Beeswarm (gráfico de enjambre)
El beeswarm dibuja **un punto por alojamiento**, acomodados uno al lado del otro para no superponerse. Así vemos dónde se concentran los datos y **cuántos** hay en cada barrio, algo que el boxplot no muestra. En seaborn se llama `swarmplot`.

Como la antigüedad toma valores enteros (0, 1, 2… años), los puntos forman "filas": el ancho de cada fila indica cuántos alojamientos hay con esa antigüedad.

⚠️ Si aparece el aviso *"X% of the points cannot be placed"*, es que no entran todos los puntos sin superponerse: agrandá la figura (`figsize`) o achicá los puntos (`size`).

In [ ]:
plt.figure(figsize=(14,7))
sns.swarmplot(data=df_top6, x="barrio", y="hosts_time_as_user_years", order=orden,
              size=3.5, hue="barrio", legend=False)

plt.title("Antigüedad del host por barrio (un punto = un alojamiento)", fontsize=14, weight="bold")
plt.xlabel("Barrio")
plt.ylabel("Años como usuario de Airbnb")
plt.tight_layout()
plt.show()

💡 Podemos **combinar** los dos: el boxplot resume y los puntos muestran los datos reales.

In [ ]:
plt.figure(figsize=(14,7))

# showfliers=False: no dibujamos los outliers del boxplot porque ya aparecen como puntos
sns.boxplot(data=df_top6, x="barrio", y="hosts_time_as_user_years", order=orden,
            color="white", showfliers=False)
sns.swarmplot(data=df_top6, x="barrio", y="hosts_time_as_user_years", order=orden,
              size=3.5, color="steelblue", alpha=0.7)

plt.title("Antigüedad del host por barrio: boxplot + beeswarm", fontsize=14, weight="bold")
plt.xlabel("Barrio")
plt.ylabel("Años como usuario de Airbnb")
plt.tight_layout()
plt.show()

✍️ **Para responder:** ¿Se ven dos grupos de hosts (nuevos y antiguos)? ¿En qué barrio se ve más claro? ¿Por qué el enjambre de Palermo es mucho más ancho que el de Belgrano?

## 3. Dormitorios × precio × barrio
Ahora combinamos **tres variables**. Antes de graficar, dos cuidados:

1. **Outliers de precio:** como en la U5, quitamos los precios atípicos con el criterio de Tukey.
2. **Pocos casos:** casi no hay alojamientos con 4 o más dormitorios, así que nos quedamos con 1 a 3 dormitorios y los 5 barrios con más alojamientos.

In [ ]:
# Criterio de Tukey sobre price_usd (U4/U5)
Q1 = df["price_usd"].quantile(0.25)
Q3 = df["price_usd"].quantile(0.75)
limite_sup = Q3 + 1.5 * (Q3 - Q1)

top5 = df["barrio"].value_counts().head(5).index

df_dorm = df[
    (df["price_usd"] <= limite_sup)
    & (df["bedrooms"] <= 3)
    & (df["barrio"].isin(top5))
].copy()

# bedrooms viene como float (1.0, 2.0...): lo pasamos a entero para que las etiquetas se vean mejor
df_dorm["bedrooms"] = df_dorm["bedrooms"].astype(int)

print(f"Límite superior de precio: USD {limite_sup:.2f}")
df["bedrooms"].value_counts().sort_index()

In [ ]:
# Precio mediano y cantidad de alojamientos por barrio y dormitorios
df_dorm_agg = df_dorm.groupby(["barrio", "bedrooms"]).agg(
    precio_mediano_usd=("price_usd", "median"),
    alojamientos=("price_usd", "count"),
).reset_index()
df_dorm_agg

### Barras agrupadas
Esta tabla ya está en formato `long` (una fila por barrio × dormitorios), así que **no hace falta `melt`**: va directo a `hue`.

In [ ]:
sns.set_theme(style="whitegrid", palette="Set2")
plt.figure(figsize=(10,6))

sns.barplot(data=df_dorm_agg, x="barrio", y="precio_mediano_usd", hue="bedrooms", palette="Set2")

plt.title("Precio mediano por barrio y cantidad de dormitorios", fontsize=14, weight="bold")
plt.xlabel("Barrio")
plt.ylabel("Precio mediano (USD)")
plt.legend(title="Dormitorios")
plt.tight_layout()
plt.show()

### Heatmap (mapa de calor)
Otra forma de mostrar tres variables: una **grilla** donde filas = barrio, columnas = dormitorios y el **color** = precio. Primero pasamos la tabla a formato `wide` con `pivot`.

In [ ]:
df_pivot = df_dorm_agg.pivot(index="barrio", columns="bedrooms", values="precio_mediano_usd")
df_pivot

In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(df_pivot, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "Precio mediano (USD)"})

plt.title("Precio mediano (USD) por barrio y dormitorios", fontsize=14, weight="bold")
plt.xlabel("Dormitorios")
plt.ylabel("")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

✍️ **Para responder:**
1. ¿En todos los barrios el precio sube con los dormitorios?
2. Mirá la columna `alojamientos` de `df_dorm_agg`: ¿qué celdas del heatmap se calcularon con muy pocos casos? ¿Confiarías en ellas?

## 4. Precio × reseñas × puntaje
Las tres variables son **numéricas**, así que no hay categorías para poner en barras. Usamos un **scatterplot**: `x` = cantidad de reseñas, `y` = precio, y el **color** = puntaje.

El puntaje está muy concentrado (la mitad de los alojamientos tiene más de 4.86), así que con una escala continua casi todos los puntos saldrían del mismo color. Lo pasamos a **rangos** con `pd.cut`.

In [ ]:
df_scatter = df[df["price_usd"] <= limite_sup].copy()

df_scatter["puntaje"] = pd.cut(
    df_scatter["review_scores"],
    bins=[0, 4.5, 4.8, 4.95, 5],
    labels=["menos de 4.5", "4.5 a 4.8", "4.8 a 4.95", "más de 4.95"],
)
df_scatter["puntaje"].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df_scatter,
    x="number_reviews",
    y="price_usd",
    hue="puntaje",
    palette="RdYlGn",   # rojo = puntaje bajo, verde = puntaje alto
    alpha=0.7,
)

plt.title("Precio vs cantidad de reseñas, coloreado por puntaje", fontsize=14, weight="bold")
plt.xlabel("Cantidad de reseñas")
plt.ylabel("Precio (USD)")
plt.legend(title="Puntaje")
plt.tight_layout()
plt.show()

💡 La mayoría de los alojamientos tiene pocas reseñas y quedan apretados a la izquierda. Con **escala logarítmica** en el eje X se distribuyen mejor.

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(data=df_scatter, x="number_reviews", y="price_usd",
                hue="puntaje", palette="RdYlGn", alpha=0.7)

plt.xscale("log")
plt.title("Precio vs cantidad de reseñas (escala log), coloreado por puntaje", fontsize=14, weight="bold")
plt.xlabel("Cantidad de reseñas (escala logarítmica)")
plt.ylabel("Precio (USD)")
plt.legend(title="Puntaje")
plt.tight_layout()
plt.show()

✍️ **Para responder:**
1. ¿Dónde aparecen los puntos rojos (puntaje bajo): con muchas o con pocas reseñas? ¿Por qué?
2. ¿Los alojamientos más caros tienen mejor puntaje?

---
# 🔑 Resumen: ¿qué gráfico uso?

| Pregunta | Tipo de variables | Gráfico |
|---|---|---|
| ¿Cuántos hay de cada categoría? | categoría → conteo | barras simple |
| ¿Cómo se compara una métrica entre categorías? | categoría → número resumen | barras simple / horizontal |
| ¿Cómo se comparan dos métricas? | categoría → 2 números | barras agrupadas |
| ¿Cómo se reparte un total? | categoría × categoría | barras apiladas (cantidad o 100%) |
| ¿Cómo se distribuye un número en cada grupo? | categoría → número | boxplot, beeswarm |
| ¿Cómo cambia un número según dos categorías? | categoría × categoría → número | barras agrupadas, heatmap |
| ¿Cómo se relacionan dos o tres números? | número × número (× número) | scatterplot con color / tamaño |